# `pfield_new` vs `pfield`: validating the nested-Param rewrite

`pfield_new.py` is a line-for-line copy of `pfield.py` with every flat
attribute access (`param.fc`, `param.Nelements`, `param.RXdelay`, ...)
replaced by the nested form (`param.xdcr.fc`, `param.xdcr.nelements`,
`param.rx.delay`, ...) introduced by the `Param`/`XdcrParams`/`MediumParams`/
`TxParams`/`RxParams` refactor. No numerical logic was changed.

Since the deprecated flat properties on `Param` read/write the *same*
underlying nested objects, a `Param` built the old way (e.g. via
`pymust.getparam`, which still sets flat attributes) should drive
`pfield_new` to the exact same result as `pfield` - this notebook checks
that directly, across a handful of distinct code paths (linear/convex
arrays, elevation focusing, baffle types, attenuation, the SIMUS branch,
NaN transmit delays).

Note: a couple of test cases below deliberately use small grids/point
counts. `pfield` has a severe, pre-existing performance cliff for certain
combinations of grid size, element count, and frequency resolution
(unrelated to this refactor) - staying small keeps this notebook fast.

In [1]:
import sys
sys.path.insert(0, "../src")

import copy
import numpy as np
import pymust
from pymust.pfield import pfield
from pymust.pfield_new import pfield_new

GRID_SIZE = 10  # kept small/fixed across cases; see note above


def assert_close(a, b, label):
    a = np.asarray(a)
    b = np.asarray(b)
    assert a.shape == b.shape, f"{label}: shape mismatch {a.shape} vs {b.shape}"
    diff = np.abs(a - b) if (np.iscomplexobj(a) or np.iscomplexobj(b)) else np.abs(a.astype(float) - b.astype(float))
    max_diff = diff.max() if diff.size else 0.0
    assert max_diff == 0.0, f"{label}: outputs differ, max abs diff = {max_diff}"
    print(f"  {label}: OK (identical, shape={a.shape})")


def run_case(name, build_param, x, y, z, delays, options_builder=None):
    print(f"--- {name} ---")
    param_a = build_param()
    param_b = copy.deepcopy(param_a)

    opts_a = options_builder() if options_builder else None
    opts_b = options_builder() if options_builder else None

    rp_a, spect_a, idx_a = pfield(x.copy(), None if y is None else y.copy(), z.copy(),
                                   delays.copy(), param_a, options=opts_a)
    rp_b, spect_b, idx_b = pfield_new(x.copy(), None if y is None else y.copy(), z.copy(),
                                       delays.copy(), param_b, options=opts_b)

    assert_close(idx_a, idx_b, "IDX")
    assert_close(rp_a, rp_b, "RP")
    assert_close(spect_a, spect_b, "SPECT")
    print("  PASSED\n")

## Test 1: linear array, 2-D, plane wave

In [2]:
def build1():
    p = pymust.getparam('L11-5v')
    p.Nelements = 16
    return p

x, z = pymust.impolgrid(GRID_SIZE, 0.04, np.pi / 4, build1())
delays = pymust.txdelayPlane(build1(), 0.1)
run_case("linear array, 2D, plane wave", build1, x, None, z, delays)

--- linear array, 2D, plane wave ---
  IDX: OK (identical, shape=(411,))
  RP: OK (identical, shape=(10, 10))
  SPECT: OK (identical, shape=(10, 10, 278))
  PASSED



## Test 2: elevation focusing (3-D, exercises the multi-Gaussian beam model path)

In [3]:
def build2():
    p = pymust.getparam('L11-5v')
    p.Nelements = 16
    p.height = 5e-3
    p.focus = 18e-3
    return p

y2 = np.full_like(x, 1e-3)
run_case("linear array, elevation focusing (3D)", build2, x, y2, z, delays)

--- linear array, elevation focusing (3D) ---
  IDX: OK (identical, shape=(411,))
  RP: OK (identical, shape=(10, 10))
  SPECT: OK (identical, shape=(10, 10, 278))
  PASSED



## Test 3: convex array (finite radius of curvature + soft-baffle obliquity factor)

In [4]:
def build3():
    p = pymust.getparam('C5-2v')
    p.Nelements = 16
    return p

x3, z3 = pymust.impolgrid(GRID_SIZE, 0.06, np.pi / 3, build3())
delays3 = pymust.txdelayPlane(build3(), 0.0)
run_case("convex array (C5-2v), 2D", build3, x3, None, z3, delays3)

--- convex array (C5-2v), 2D ---
  IDX: OK (identical, shape=(285,))
  RP: OK (identical, shape=(10, 10))
  SPECT: OK (identical, shape=(10, 10, 193))
  PASSED



## Test 4: rigid baffle + nonzero attenuation

In [5]:
def build4():
    p = pymust.getparam('L11-5v')
    p.Nelements = 16
    p.baffle = 'rigid'
    p.attenuation = 0.5
    return p

run_case("linear array, rigid baffle + attenuation", build4, x, None, z, delays)

--- linear array, rigid baffle + attenuation ---
  IDX: OK (identical, shape=(411,))
  RP: OK (identical, shape=(10, 10))
  SPECT: OK (identical, shape=(10, 10, 278))
  PASSED



## Test 5: SIMUS-style call (the `isSIMUS` branch: RC, RXdelay, CallFun)

This mimics how `simus.py` actually calls `pfield` internally (received-signal
spectrum, RC-weighted scatterers, RXdelay applied). Uses a tiny point count -
see the performance note in the intro.

In [6]:
def build5():
    p = pymust.getparam('L11-5v')
    p.Nelements = 8
    p.RXdelay = np.zeros((1, 8), dtype=np.float32)
    return p

x5 = np.array([[0.0, 0.005], [0.01, -0.005]])
z5 = np.array([[0.02, 0.025], [0.03, 0.02]])
delays5 = pymust.txdelayPlane(build5(), 0.05)

def opts5():
    o = pymust.utils.Options()
    o.CallFun = 'simus'
    o.RC = np.ones((x5.size,), dtype=np.float32)
    o.dBThresh = -60
    # For isSIMUS/isMKMOVIE, pfield reads df directly from options.FrequencyStep
    # as a physical Hz value (simus.py normally pre-computes and overwrites this
    # before calling pfield). Leaving it at the generic scaling-factor default of
    # 1 makes Nf ~ 2*fc/1 (~15 million samples) here - not a pfield bug, just an
    # unrealistic df for a direct call that bypasses simus.py's own setup.
    o.FrequencyStep = 5e4
    return o

run_case("simus-style call (isSIMUS branch)", build5, x5, None, z5, delays5, opts5)

--- simus-style call (isSIMUS branch) ---
  IDX: OK (identical, shape=(305,))
  RP: OK (identical, shape=())
  SPECT: OK (identical, shape=(205, 8))
  PASSED



## Test 6: NaN transmit delays (exercises the transmit-apodization zeroing branch)

In [7]:
def build6():
    p = pymust.getparam('L11-5v')
    p.Nelements = 16
    return p

delays6 = delays.copy()
delays6[0, 0] = np.nan
delays6[0, 3] = np.nan
run_case("NaN transmit delays (apodization zeroing)", build6, x, None, z, delays6)

--- NaN transmit delays (apodization zeroing) ---
  IDX: OK (identical, shape=(411,))
  RP: OK (identical, shape=(10, 10))
  SPECT: OK (identical, shape=(10, 10, 278))
  PASSED



## Summary

In [8]:
print("ALL PFIELD vs PFIELD_NEW COMPARISONS PASSED - outputs are bit-identical")

ALL PFIELD vs PFIELD_NEW COMPARISONS PASSED - outputs are bit-identical
